In [7]:
!pip install -q transformers accelerate sentencepiece pandas tqdm


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [11]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Output_Generation.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/training_pairs-checkpoint.jsonl
	.ipynb_checkpoints/training_pairs_rocm-checkpoint.jsonl
	training_pairs.jsonl
	training_pairs_rocm.jsonl

no changes added to commit (use "git add" and/or "git commit -a")


In [5]:
!git add .

In [9]:
!git commit -m "sample training data prepared"

[main 8b47ea5] sample training data prepared
 4 files changed, 646 insertions(+)
 create mode 100644 .ipynb_checkpoints/Output_Generation-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/training_pairs_50-checkpoint.jsonl
 create mode 100644 Output_Generation.ipynb
 create mode 100644 training_pairs_50.jsonl


In [8]:
!git config --global user.name "ChaitanyaParab11"
!git config --global user.email "chaitanyaparab111@gmail.com"

In [ ]:
!git push origin main

Username for 'https://github.com': 

In [ ]:
import json
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

print("Model loaded")

# ---------------------------------------
# LOAD DATA
# ---------------------------------------

df = pd.read_csv("data/train_vulnerable.csv")

# PILOT RUN
# df = df.head(50)

print("Samples:", len(df))

# ---------------------------------------
# GENERATE FIXES
# ---------------------------------------

output_file = "training_pairs.jsonl"

with open(output_file, "w", encoding="utf-8") as outfile:

    for idx, row in tqdm(df.iterrows(), total=len(df)):

        category = str(row["category"])
        cwe = str(row["cwe"])
        code = str(row["source_code"])

        prompt = f"""
You are a senior Java security engineer.

Vulnerability Category: {category}
CWE: CWE-{cwe}

Fix the security vulnerability.

Requirements:
1. Preserve functionality.
2. Use secure coding practices.
3. Return ONLY complete corrected Java code.
4. No markdown.
5. No explanations.

Java Code:

{code}
"""

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=12000
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=2500,
            do_sample=False,
            temperature=0.0
        )

        generated_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        generated_text = generated_text.replace(
            "```java",
            ""
        )
        
        generated_text = generated_text.replace(
            "```",
            ""
        )
        
        generated_text = generated_text.strip()

        record = {
            "instruction": f"Fix CWE-{cwe} {category} vulnerability",
            "input": code,
            "output": generated_text
        }

        outfile.write(
            json.dumps(record, ensure_ascii=False)
            + "\n"
        )

        if (idx + 1) % 5 == 0:
            print(f"Completed {idx + 1}")

print("Done")
print("Saved:", output_file)